                                        ## Ranker Training Dataset for Recommendation System
This ranker training dataset enables the development of an intelligent recommendation system by combining collaborative filtering, content-based signals, contextual information, and embeddings into a unified learning framework.
## Objective
To construct a supervised learning dataset where candidate items are ranked based on multiple behavioral, contextual, and semantic features, enabling the system to predict the most relevant items for a user.
## Data Sources
The system uses three main datasets:
Basket dataset → customer purchase history
Category embedding lookup → vector representation of categories
Item catalog → mapping between item and category
## Data Processing
Basket item sets are parsed into structured lists
Category sets are cleaned and normalized
Embedding strings are converted into numerical vectors
Unique basket identifiers are created
## Data Transformation (Exploded View)
Each basket is transformed into item-level rows:
One row per item per basket
Includes contextual features such as season, time, and holidays
## Item Co-occurrence Matrix
A co-occurrence structure is built to capture how often items appear together in baskets.
This helps identify relationships between items (e.g., items frequently bought together).
## Context-Based Frequency
A context-aware mapping is created based on:
Season
Holiday/Festival
Time slot
Week of the month
This captures which items are popular under specific contexts.
## User Behavior Modeling
User-level statistics are computed:
User → item frequency
User → category frequency
This enables personalization based on past behavior.
## Category Embedding Integration
Category embeddings are used to represent semantic relationships.
Similarity between basket context and candidate items is computed using cosine similarity.
## Candidate Generation
Candidate items are generated from multiple sources:
Co-purchase items (collaborative filtering)
Context-based popular items
User history items
These candidates are merged to form a relevant candidate pool.
## Feature Engineering
For each candidate item, multiple features are computed:
Item co-occurrence score
Customer history score
Category affinity score
Context popularity score
Embedding similarity score
Basket size and contextual attributes
These features represent different signals used for ranking.
## Label Construction
Positive samples (label = 1) → items actually purchased
Negative samples (label = 0) → candidate items not purchased
This converts the problem into a supervised ranking task.
## Grouping for Ranking
Each set of candidates is grouped using a unique group ID based on context.
This allows ranking models to compare items within the same context group.

## Dataset Generation
The system iterates through all baskets and:
Generates candidate items
Creates feature vectors
Assigns labels
Builds the final training dataset
## Output Dataset
The final dataset contains:
Customer ID
Candidate item ID and category
Feature columns
Label (target)
Group ID
This dataset is saved as a CSV file for model training.

## Feature Definition
Key feature columns include:
basket_size
item_cooccurrence_score
category_affinity_score
context_popularity_score
customer_history_score
embedding_similarity_score
isHoliday, isFestival, weekOfMonth
## Recommendation Flow
Generate candidate items for a user
Compute features for each candidate
Apply trained ranking model
Sort candidates based on predicted relevance
Recommend top-N items

Import Libraries

In [1]:
import json
import ast
import math
import random
import re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

Load Input Datasets

In [2]:
DATA_DIR = Path(r"D:\recommendation_item_API\data")

BASKET_FILE = DATA_DIR / "customer_purchase_pattern_history_final.csv"
CATEGORY_LOOKUP_FILE = DATA_DIR / "category_embedding_lookup_from_customer_history.csv"
ITEM_CATALOG_FILE = DATA_DIR / "item_catalog.csv"

print("Basket file exists:", BASKET_FILE.exists())
print("Category lookup file exists:", CATEGORY_LOOKUP_FILE.exists())
print("Item catalog file exists:", ITEM_CATALOG_FILE.exists())

Basket file exists: True
Category lookup file exists: True
Item catalog file exists: True


Helper Functions

In [3]:
basket_df = pd.read_csv(BASKET_FILE)
cat_lookup_df = pd.read_csv(CATEGORY_LOOKUP_FILE)
item_catalog_df = pd.read_csv(ITEM_CATALOG_FILE)

basket_df.columns = [c.strip() for c in basket_df.columns]
cat_lookup_df.columns = [c.strip() for c in cat_lookup_df.columns]
item_catalog_df.columns = [c.strip() for c in item_catalog_df.columns]

print("basket_df shape:", basket_df.shape)
print("cat_lookup_df shape:", cat_lookup_df.shape)
print("item_catalog_df shape:", item_catalog_df.shape)

display(basket_df.head())
display(cat_lookup_df.head())
display(item_catalog_df.head())

basket_df shape: (6108, 14)
cat_lookup_df shape: (43, 10)
item_catalog_df shape: (229, 10)


,customerId,purchaseDate,categorySet,itemIdSet,itemNameSet,season,seasonLabel,isHoliday,isFestival,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth,basket_category_embedding
0,23561,01-01-25,"{Dairy-Other, Bakery-Bread, Beverage-Hot}","{7075, 27, 6814}","{Herman Peanut Butter-340gm, Pran Toast-250g, ...",Winter,1,0,0,Morning,1,1,1,"[29.928769, -0.264688, -0.632273, -0.389245, 0..."
1,23569,01-01-25,"{Pantry-DryFood, Fruits-Fresh, Bakery-Bread, D...","{25474, 15131, 13786, 7017, 6815, 31849, 13352}","{Lassa Special Semai-500g(Pkt), Atta Fruits, C...",Winter,1,0,0,Morning,1,1,1,"[28.234124, 0.153048, -0.106894, -0.015198, 0...."
2,23527,01-01-25,"{Pantry-Oils, Household-Cleaning, Fish-Fresh, ...","{32441, 2306, 13922, 2364, 952, 30743, 32269}","{Ela vista pomace olive oil 250 ml, Harpic Tot...",Winter,1,0,0,Afternoon,3,1,1,"[30.191856, -0.558436, 0.644018, 0.44081, -0.7..."
3,23433,01-01-25,"{Spices-Cooking, Veg-Cooking, Pantry-DryFood}","{878, 704, 15104, 31328, 14964}","{Radhuni Spices (Corinder)-100gm, Radhuni Spic...",Winter,1,0,0,Afternoon,3,1,1,"[31.248554, -0.560331, -0.796425, -1.717897, -..."
4,23494,01-01-25,"{Pantry-Sweeteners, Personal-Care-Oral, Househ...","{15180, 2220, 2364, 13793, 708, 2043, 12655}","{Akher Sugar -1kg, Ambition Tooth brush, Rok T...",Winter,1,0,0,Afternoon,3,1,1,"[29.365952, 0.023257, -0.854166, -0.450602, 0...."


,category,cat_emb_0,cat_emb_1,cat_emb_2,cat_emb_3,cat_emb_4,cat_emb_5,cat_emb_6,cat_emb_7,category_embedding
0,Bakery-Bread,30.500512,-0.312345,-1.336538,0.171504,1.265434,-0.221737,-2.422268,-1.024791,"[30.500512, -0.312345, -1.336538, 0.171504, 1...."
1,Beverage-Carbonated,30.793406,0.582054,2.320642,-1.194838,0.826971,-1.544423,1.249234,-0.971547,"[30.793406, 0.582054, 2.320642, -1.194838, 0.8..."
2,Beverage-Hot,29.132634,0.224027,0.067448,-0.667184,-0.053071,0.364866,0.603695,0.191561,"[29.132634, 0.224027, 0.067448, -0.667184, -0...."
3,Beverage-Juice,31.321391,0.378736,0.439785,-0.637943,0.784723,-1.583087,-2.383535,2.169788,"[31.321391, 0.378736, 0.439785, -0.637943, 0.7..."
4,Beverage-Water,22.344446,0.596863,0.331393,-0.024937,0.778449,0.414786,-0.005348,-0.207390,"[22.344446, 0.596863, 0.331393, -0.024937, 0.7..."


,itemId,itemName,category,avgPrice,minPrice,maxPrice,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,121.34,121.34,121.34,115,1,1,0
1,27,Pran Toast-250g,Bakery-Bread,63.66,63.66,63.66,244,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Personal-Care-Cosmetics,587.26,587.26,587.26,65,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,122.60,122.60,122.60,43,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,73.72,73.72,73.72,144,1,1,0


 Basket Data Preprocessing


In [4]:
def parse_set_like(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    if value == "":
        return []
    
    if value.startswith("{") and value.endswith("}"):
        value = value[1:-1].strip()
    
    if value == "":
        return []
    
    parts = [x.strip() for x in value.split(",")]
    parts = [x for x in parts if x != ""]
    return parts


def parse_numeric_set(value):
    parts = parse_set_like(value)
    result = []
    for x in parts:
        try:
            result.append(int(float(x)))
        except:
            pass
    return result


def parse_json_vector(value):
    if pd.isna(value):
        return None
    try:
        if isinstance(value, str):
            arr = ast.literal_eval(value)
        else:
            arr = value
        return np.array(arr, dtype=float)
    except:
        return None


def normalize_text(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x


def normalize_category(cat):
    if pd.isna(cat):
        return cat
    cat = normalize_text(cat)
    parts = [p.strip() for p in cat.split("-")]
    parts = [p.capitalize() if p else p for p in parts]
    return "-".join(parts)


def cosine_sim(vec1, vec2):
    if vec1 is None or vec2 is None:
        return 0.0
    
    v1 = np.array(vec1, dtype=float).reshape(1, -1)
    v2 = np.array(vec2, dtype=float).reshape(1, -1)
    
    if np.allclose(v1, 0) or np.allclose(v2, 0):
        return 0.0
    
    return float(cosine_similarity(v1, v2)[0][0])

Category Embedding 

In [5]:
basket_df["itemIdSet_list"] = basket_df["itemIdSet"].apply(parse_numeric_set)
basket_df["categorySet_list"] = basket_df["categorySet"].apply(parse_set_like)
basket_df["categorySet_list"] = basket_df["categorySet_list"].apply(
    lambda cats: [normalize_category(c) for c in cats]
)

basket_df["basket_category_embedding_vec"] = basket_df["basket_category_embedding"].apply(parse_json_vector)

# basket id তৈরি করছি
basket_df = basket_df.reset_index(drop=True)
basket_df["basket_id"] = basket_df.index + 1

display(
    basket_df[
        ["basket_id", "customerId", "itemIdSet_list", "categorySet_list", "basket_category_embedding_vec"]
    ].head()
)

,basket_id,customerId,itemIdSet_list,categorySet_list,basket_category_embedding_vec
0,1,23561,"[7075, 27, 6814]","[Dairy-Other, Bakery-Bread, Beverage-Hot]","[29.928769, -0.264688, -0.632273, -0.389245, 0..."
1,2,23569,"[25474, 15131, 13786, 7017, 6815, 31849, 13352]","[Pantry-Dryfood, Fruits-Fresh, Bakery-Bread, D...","[28.234124, 0.153048, -0.106894, -0.015198, 0...."
2,3,23527,"[32441, 2306, 13922, 2364, 952, 30743, 32269]","[Pantry-Oils, Household-Cleaning, Fish-Fresh, ...","[30.191856, -0.558436, 0.644018, 0.44081, -0.7..."
3,4,23433,"[878, 704, 15104, 31328, 14964]","[Spices-Cooking, Veg-Cooking, Pantry-Dryfood]","[31.248554, -0.560331, -0.796425, -1.717897, -..."
4,5,23494,"[15180, 2220, 2364, 13793, 708, 2043, 12655]","[Pantry-Sweeteners, Personal-Care-Oral, Househ...","[29.365952, 0.023257, -0.854166, -0.450602, 0...."


In [6]:
cat_lookup_df["category"] = cat_lookup_df["category"].apply(normalize_category)
cat_lookup_df["category_embedding_vec"] = cat_lookup_df["category_embedding"].apply(parse_json_vector)

category_to_vector = dict(zip(cat_lookup_df["category"], cat_lookup_df["category_embedding_vec"]))

print("Total categories in lookup:", len(category_to_vector))

sample_key = list(category_to_vector.keys())[0]
print("Sample category:", sample_key)
print("Sample vector:", category_to_vector[sample_key])

Total categories in lookup: 43
Sample category: Bakery-Bread
Sample vector: [30.500512 -0.312345 -1.336538  0.171504  1.265434 -0.221737 -2.422268
 -1.024791]


Item Catalog Mapping

In [7]:
if "itemId" not in item_catalog_df.columns or "category" not in item_catalog_df.columns:
    raise ValueError("item_catalog.csv এ itemId আর category column থাকতে হবে")

item_catalog_df["itemId"] = item_catalog_df["itemId"].astype(int)
item_catalog_df["category"] = item_catalog_df["category"].apply(normalize_category)

item_to_category = dict(zip(item_catalog_df["itemId"], item_catalog_df["category"]))

print("Total items in catalog:", len(item_to_category))
item_catalog_df.head()

Total items in catalog: 229


,itemId,itemName,category,avgPrice,minPrice,maxPrice,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,121.34,121.34,121.34,115,1,1,0
1,27,Pran Toast-250g,Bakery-Bread,63.66,63.66,63.66,244,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Personal-Care-Cosmetics,587.26,587.26,587.26,65,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,122.60,122.60,122.60,43,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,73.72,73.72,73.72,144,1,1,0


Explode Basket into Row-Level Data

In [8]:
exploded_rows = []

for _, row in basket_df.iterrows():
    basket_id = row["basket_id"]
    customer_id = row["customerId"]
    purchase_date = row["purchaseDate"]
    season = row["season"]
    season_label = row["seasonLabel"]
    is_holiday = row["isHoliday"]
    is_festival = row["isFestival"]
    time_slot = row["timeSlot"]
    time_slot_label = row["timeSlotLabel"]
    month_part = row["monthPartLabel"]
    week_of_month = row["weekOfMonth"]
    
    for item_id in row["itemIdSet_list"]:
        exploded_rows.append({
            "basket_id": basket_id,
            "customerId": customer_id,
            "purchaseDate": purchase_date,
            "itemId": item_id,
            "category": item_to_category.get(item_id, None),
            "season": season,
            "seasonLabel": season_label,
            "isHoliday": is_holiday,
            "isFestival": is_festival,
            "timeSlot": time_slot,
            "timeSlotLabel": time_slot_label,
            "monthPartLabel": month_part,
            "weekOfMonth": week_of_month
        })

exploded_df = pd.DataFrame(exploded_rows)

print("Exploded shape:", exploded_df.shape)
display(exploded_df.head())

Exploded shape: (40886, 13)


,basket_id,customerId,purchaseDate,itemId,category,season,seasonLabel,isHoliday,isFestival,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth
0,1,23561,01-01-25,7075,Dairy-Other,Winter,1,0,0,Morning,1,1,1
1,1,23561,01-01-25,27,Bakery-Bread,Winter,1,0,0,Morning,1,1,1
2,1,23561,01-01-25,6814,Beverage-Hot,Winter,1,0,0,Morning,1,1,1
3,2,23569,01-01-25,25474,Pantry-Dryfood,Winter,1,0,0,Morning,1,1,1
4,2,23569,01-01-25,15131,Fruits-Fresh,Winter,1,0,0,Morning,1,1,1


Item Co-occurrence Calculation

In [9]:
item_pair_counter = Counter()

basket_item_lists = basket_df[["basket_id", "itemIdSet_list"]].copy()

for _, row in basket_item_lists.iterrows():
    items = sorted(set(row["itemIdSet_list"]))
    n = len(items)
    
    for i in range(n):
        for j in range(i + 1, n):
            a = items[i]
            b = items[j]
            item_pair_counter[(a, b)] += 1
            item_pair_counter[(b, a)] += 1

print("Total directional item pairs:", len(item_pair_counter))
print("Sample:", list(item_pair_counter.items())[:10])

Total directional item pairs: 37790
Sample: [((27, 6814), 39), ((6814, 27), 39), ((27, 7075), 7), ((7075, 27), 7), ((6814, 7075), 5), ((7075, 6814), 5), ((6815, 7017), 9), ((7017, 6815), 9), ((6815, 13352), 44), ((13352, 6815), 44)]


In [10]:
context_item_counter = Counter()

for _, row in exploded_df.iterrows():
    context_key = (
        row["season"],
        int(row["isHoliday"]),
        int(row["isFestival"]),
        row["timeSlot"],
        int(row["weekOfMonth"])
    )
    context_item_counter[(context_key, row["itemId"])] += 1

print("Total context item pairs:", len(context_item_counter))
print("Sample:", list(context_item_counter.items())[:10])

Total context item pairs: 19199
Sample: [((('Winter', 0, 0, 'Morning', 1), 7075), 4), ((('Winter', 0, 0, 'Morning', 1), 27), 6), ((('Winter', 0, 0, 'Morning', 1), 6814), 6), ((('Winter', 0, 0, 'Morning', 1), 25474), 2), ((('Winter', 0, 0, 'Morning', 1), 15131), 1), ((('Winter', 0, 0, 'Morning', 1), 13786), 4), ((('Winter', 0, 0, 'Morning', 1), 7017), 4), ((('Winter', 0, 0, 'Morning', 1), 6815), 1), ((('Winter', 0, 0, 'Morning', 1), 31849), 2), ((('Winter', 0, 0, 'Morning', 1), 13352), 13)]


In [11]:
user_item_counter = Counter()
user_category_counter = Counter()

for _, row in exploded_df.iterrows():
    user_id = row["customerId"]
    item_id = row["itemId"]
    category = row["category"]
    
    user_item_counter[(user_id, item_id)] += 1
    
    if pd.notna(category):
        user_category_counter[(user_id, category)] += 1

print("User item history size:", len(user_item_counter))
print("User category history size:", len(user_category_counter))

User item history size: 10506
User category history size: 3956


In [12]:
def get_candidate_item_category(item_id, item_to_category):
    return item_to_category.get(int(item_id), None)


def get_candidate_item_category_vector(item_id, item_to_category, category_to_vector):
    category = get_candidate_item_category(item_id, item_to_category)
    if category is None:
        return None
    return category_to_vector.get(category, None)

In [13]:
def get_embedding_similarity_for_candidate(basket_embedding_vec, candidate_item_id, item_to_category, category_to_vector):
    candidate_vec = get_candidate_item_category_vector(candidate_item_id, item_to_category, category_to_vector)
    return cosine_sim(basket_embedding_vec, candidate_vec)

In [14]:
sample_row = basket_df.iloc[0]

sample_basket_vec = sample_row["basket_category_embedding_vec"]
sample_item_ids = sample_row["itemIdSet_list"]

if len(sample_item_ids) > 0:
    sample_candidate = sample_item_ids[0]
    sim_score = get_embedding_similarity_for_candidate(
        sample_basket_vec,
        sample_candidate,
        item_to_category,
        category_to_vector
    )
    print("Sample candidate:", sample_candidate)
    print("Similarity score:", sim_score)

Sample candidate: 7075
Similarity score: 0.9964860481194298


In [15]:
def mean_pool_vectors(vectors):
    valid_vectors = [np.array(v, dtype=float) for v in vectors if v is not None]
    
    if len(valid_vectors) == 0:
        return None
    
    return np.mean(valid_vectors, axis=0)


def build_context_basket_embedding_from_items(context_item_ids, item_to_category, category_to_vector):
    context_categories = []
    
    for item_id in context_item_ids:
        cat = item_to_category.get(int(item_id), None)
        if cat is not None:
            context_categories.append(cat)
    
    context_categories = list(dict.fromkeys(context_categories))
    
    vectors = [category_to_vector.get(cat, None) for cat in context_categories]
    return mean_pool_vectors(vectors)

In [16]:
def get_item_cooccurrence_score(candidate_item_id, context_item_ids, item_pair_counter):
    if len(context_item_ids) == 0:
        return 0.0
    
    scores = []
    for ctx_item in context_item_ids:
        scores.append(item_pair_counter.get((int(candidate_item_id), int(ctx_item)), 0))
    
    if len(scores) == 0:
        return 0.0
    
    return float(np.mean(scores))

In [17]:
def get_customer_history_score(customer_id, candidate_item_id, user_item_counter):
    return float(user_item_counter.get((customer_id, int(candidate_item_id)), 0))

In [18]:
def get_category_affinity_score(customer_id, candidate_item_id, item_to_category, user_category_counter):
    category = item_to_category.get(int(candidate_item_id), None)
    if category is None:
        return 0.0
    
    return float(user_category_counter.get((customer_id, category), 0))

In [19]:
def get_context_popularity_score(row_context_key, candidate_item_id, context_item_counter):
    return float(context_item_counter.get((row_context_key, int(candidate_item_id)), 0))

In [20]:
def get_top_copurchase_candidates(context_item_ids, item_pair_counter, top_n=20):
    score_counter = Counter()
    
    for ctx_item in context_item_ids:
        for (a, b), cnt in item_pair_counter.items():
            if b == int(ctx_item):
                score_counter[a] += cnt
    
    return [item for item, _ in score_counter.most_common(top_n)]


def get_top_context_candidates(context_key, context_item_counter, top_n=20):
    score_counter = Counter()
    
    for (ctx_key, item_id), cnt in context_item_counter.items():
        if ctx_key == context_key:
            score_counter[item_id] += cnt
    
    return [item for item, _ in score_counter.most_common(top_n)]


def get_top_user_history_candidates(customer_id, user_item_counter, top_n=20):
    score_counter = Counter()
    
    for (uid, item_id), cnt in user_item_counter.items():
        if uid == customer_id:
            score_counter[item_id] += cnt
    
    return [item for item, _ in score_counter.most_common(top_n)]

In [21]:
def build_candidate_pool(customer_id, context_item_ids, context_key,
                         item_pair_counter, context_item_counter, user_item_counter,
                         top_n_each=20):
    
    c1 = get_top_copurchase_candidates(context_item_ids, item_pair_counter, top_n=top_n_each)
    c2 = get_top_context_candidates(context_key, context_item_counter, top_n=top_n_each)
    c3 = get_top_user_history_candidates(customer_id, user_item_counter, top_n=top_n_each)
    
    merged = []
    seen = set()
    
    for item in c1 + c2 + c3:
        if item not in seen:
            seen.add(item)
            merged.append(item)
    
    return merged

In [22]:
def build_ranker_row(customer_id,
                     context_item_ids,
                     candidate_item_id,
                     label,
                     season,
                     is_holiday,
                     is_festival,
                     time_slot,
                     week_of_month,
                     item_pair_counter,
                     context_item_counter,
                     user_item_counter,
                     user_category_counter,
                     item_to_category,
                     category_to_vector):
    
    context_key = (
        season,
        int(is_holiday),
        int(is_festival),
        time_slot,
        int(week_of_month)
    )
    
    context_embedding_vec = build_context_basket_embedding_from_items(
        context_item_ids,
        item_to_category,
        category_to_vector
    )
    
    item_co_score = get_item_cooccurrence_score(candidate_item_id, context_item_ids, item_pair_counter)
    history_score = get_customer_history_score(customer_id, candidate_item_id, user_item_counter)
    category_affinity = get_category_affinity_score(customer_id, candidate_item_id, item_to_category, user_category_counter)
    context_popularity = get_context_popularity_score(context_key, candidate_item_id, context_item_counter)
    embedding_similarity = get_embedding_similarity_for_candidate(
        context_embedding_vec,
        candidate_item_id,
        item_to_category,
        category_to_vector
    )
    
    candidate_category = item_to_category.get(int(candidate_item_id), None)
    
    return {
        "customerId": customer_id,
        "candidate_item_id": int(candidate_item_id),
        "candidate_category": candidate_category,
        "basket_size": len(context_item_ids),
        "item_cooccurrence_score": item_co_score,
        "category_affinity_score": category_affinity,
        "context_popularity_score": context_popularity,
        "customer_history_score": history_score,
        "embedding_similarity_score": embedding_similarity,
        "season": season,
        "isHoliday": int(is_holiday),
        "isFestival": int(is_festival),
        "timeSlot": time_slot,
        "weekOfMonth": int(week_of_month),
        "label": int(label)
    }

In [23]:
training_rows = []

max_baskets = len(basket_df)       # চাইলে কমাও
negative_per_positive = 10         # শুরুতে 10 ভালো

for _, row in basket_df.iloc[:max_baskets].iterrows():
    basket_items = list(dict.fromkeys(row["itemIdSet_list"]))
    
    # খুব ছোট basket skip
    if len(basket_items) < 2:
        continue
    
    customer_id = row["customerId"]
    season = row["season"]
    is_holiday = row["isHoliday"]
    is_festival = row["isFestival"]
    time_slot = row["timeSlot"]
    week_of_month = row["weekOfMonth"]
    
    for target_item in basket_items:
        context_item_ids = [x for x in basket_items if x != target_item]
        
        context_key = (
            season,
            int(is_holiday),
            int(is_festival),
            time_slot,
            int(week_of_month)
        )
        
        candidate_pool = build_candidate_pool(
            customer_id=customer_id,
            context_item_ids=context_item_ids,
            context_key=context_key,
            item_pair_counter=item_pair_counter,
            context_item_counter=context_item_counter,
            user_item_counter=user_item_counter,
            top_n_each=20
        )
        
        # positive ensure
        if target_item not in candidate_pool:
            candidate_pool.append(target_item)
        
        # context items বাদ
        candidate_pool = [x for x in candidate_pool if x not in context_item_ids]
        
        # positive row
        pos_row = build_ranker_row(
            customer_id=customer_id,
            context_item_ids=context_item_ids,
            candidate_item_id=target_item,
            label=1,
            season=season,
            is_holiday=is_holiday,
            is_festival=is_festival,
            time_slot=time_slot,
            week_of_month=week_of_month,
            item_pair_counter=item_pair_counter,
            context_item_counter=context_item_counter,
            user_item_counter=user_item_counter,
            user_category_counter=user_category_counter,
            item_to_category=item_to_category,
            category_to_vector=category_to_vector
        )
        
        training_rows.append(pos_row)
        
        # negative candidates
        negative_candidates = [x for x in candidate_pool if x != target_item]
        
        # random sample
        if len(negative_candidates) > negative_per_positive:
            negative_candidates = random.sample(negative_candidates, negative_per_positive)
        
        for neg_item in negative_candidates:
            neg_row = build_ranker_row(
                customer_id=customer_id,
                context_item_ids=context_item_ids,
                candidate_item_id=neg_item,
                label=0,
                season=season,
                is_holiday=is_holiday,
                is_festival=is_festival,
                time_slot=time_slot,
                week_of_month=week_of_month,
                item_pair_counter=item_pair_counter,
                context_item_counter=context_item_counter,
                user_item_counter=user_item_counter,
                user_category_counter=user_category_counter,
                item_to_category=item_to_category,
                category_to_vector=category_to_vector
            )
            
            training_rows.append(neg_row)

ranker_df = pd.DataFrame(training_rows)

print("Ranker dataset shape:", ranker_df.shape)
display(ranker_df.head())

Ranker dataset shape: (449680, 15)


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1
1,23561,976,Pantry-Rice,2,20.0,3.0,2.0,0.0,0.979893,Winter,0,0,Morning,1,0
2,23561,3185,Dairy-Milk,2,25.5,2.0,0.0,0.0,0.994473,Winter,0,0,Morning,1,0
3,23561,7510,Snacks-General,2,3.5,17.0,1.0,6.0,0.987187,Winter,0,0,Morning,1,0
4,23561,12625,Pantry-Oils,2,2.0,6.0,0.0,4.0,0.990649,Winter,0,0,Morning,1,0


In [24]:
ranker_df = ranker_df.reset_index(drop=True)
ranker_df["group_id"] = (
    ranker_df["customerId"].astype(str) + "_" +
    ranker_df["season"].astype(str) + "_" +
    ranker_df["timeSlot"].astype(str) + "_" +
    ranker_df["weekOfMonth"].astype(str) + "_" +
    ranker_df.index.astype(str)
)

display(ranker_df.head())

,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,23561_Winter_Morning_1_0
1,23561,976,Pantry-Rice,2,20.0,3.0,2.0,0.0,0.979893,Winter,0,0,Morning,1,0,23561_Winter_Morning_1_1
2,23561,3185,Dairy-Milk,2,25.5,2.0,0.0,0.0,0.994473,Winter,0,0,Morning,1,0,23561_Winter_Morning_1_2
3,23561,7510,Snacks-General,2,3.5,17.0,1.0,6.0,0.987187,Winter,0,0,Morning,1,0,23561_Winter_Morning_1_3
4,23561,12625,Pantry-Oils,2,2.0,6.0,0.0,4.0,0.990649,Winter,0,0,Morning,1,0,23561_Winter_Morning_1_4


In [25]:
training_rows = []

max_baskets = len(basket_df)
negative_per_positive = 10
group_counter = 0

for _, row in basket_df.iloc[:max_baskets].iterrows():
    basket_items = list(dict.fromkeys(row["itemIdSet_list"]))
    
    if len(basket_items) < 2:
        continue
    
    customer_id = row["customerId"]
    season = row["season"]
    is_holiday = row["isHoliday"]
    is_festival = row["isFestival"]
    time_slot = row["timeSlot"]
    week_of_month = row["weekOfMonth"]
    
    for target_item in basket_items:
        context_item_ids = [x for x in basket_items if x != target_item]
        
        context_key = (
            season,
            int(is_holiday),
            int(is_festival),
            time_slot,
            int(week_of_month)
        )
        
        group_counter += 1
        group_id = group_counter
        
        candidate_pool = build_candidate_pool(
            customer_id=customer_id,
            context_item_ids=context_item_ids,
            context_key=context_key,
            item_pair_counter=item_pair_counter,
            context_item_counter=context_item_counter,
            user_item_counter=user_item_counter,
            top_n_each=20
        )
        
        if target_item not in candidate_pool:
            candidate_pool.append(target_item)
        
        candidate_pool = [x for x in candidate_pool if x not in context_item_ids]
        
        pos_row = build_ranker_row(
            customer_id=customer_id,
            context_item_ids=context_item_ids,
            candidate_item_id=target_item,
            label=1,
            season=season,
            is_holiday=is_holiday,
            is_festival=is_festival,
            time_slot=time_slot,
            week_of_month=week_of_month,
            item_pair_counter=item_pair_counter,
            context_item_counter=context_item_counter,
            user_item_counter=user_item_counter,
            user_category_counter=user_category_counter,
            item_to_category=item_to_category,
            category_to_vector=category_to_vector
        )
        pos_row["group_id"] = group_id
        training_rows.append(pos_row)
        
        negative_candidates = [x for x in candidate_pool if x != target_item]
        
        if len(negative_candidates) > negative_per_positive:
            negative_candidates = random.sample(negative_candidates, negative_per_positive)
        
        for neg_item in negative_candidates:
            neg_row = build_ranker_row(
                customer_id=customer_id,
                context_item_ids=context_item_ids,
                candidate_item_id=neg_item,
                label=0,
                season=season,
                is_holiday=is_holiday,
                is_festival=is_festival,
                time_slot=time_slot,
                week_of_month=week_of_month,
                item_pair_counter=item_pair_counter,
                context_item_counter=context_item_counter,
                user_item_counter=user_item_counter,
                user_category_counter=user_category_counter,
                item_to_category=item_to_category,
                category_to_vector=category_to_vector
            )
            neg_row["group_id"] = group_id
            training_rows.append(neg_row)

ranker_df = pd.DataFrame(training_rows)

print("Final ranker dataset shape:", ranker_df.shape)
display(ranker_df.head(20))

Final ranker dataset shape: (449680, 16)


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,13348,Bakery-Bread,2,16.5,6.0,5.0,1.0,0.997975,Winter,0,0,Morning,1,0,1
2,23561,12625,Pantry-Oils,2,2.0,6.0,0.0,4.0,0.990649,Winter,0,0,Morning,1,0,1
3,23561,976,Pantry-Rice,2,20.0,3.0,2.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
4,23561,13352,Protein-Egg,2,32.0,4.0,13.0,2.0,0.998886,Winter,0,0,Morning,1,0,1
5,23561,30745,Personal-Care-Sanitary,2,13.0,1.0,3.0,0.0,0.998238,Winter,0,0,Morning,1,0,1
6,23561,30626,Pantry-Flour,2,9.5,6.0,3.0,0.0,0.998011,Winter,0,0,Morning,1,0,1
7,23561,6812,Beverage-Hot,2,18.0,4.0,5.0,1.0,0.997758,Winter,0,0,Morning,1,0,1
8,23561,3185,Dairy-Milk,2,25.5,2.0,0.0,0.0,0.994473,Winter,0,0,Morning,1,0,1
9,23561,25474,Pantry-Dryfood,2,8.5,4.0,2.0,4.0,0.993149,Winter,0,0,Morning,1,0,1


In [26]:
print(ranker_df.columns.tolist())
print(ranker_df["label"].value_counts(dropna=False))
print("Unique groups:", ranker_df["group_id"].nunique())

['customerId', 'candidate_item_id', 'candidate_category', 'basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'season', 'isHoliday', 'isFestival', 'timeSlot', 'weekOfMonth', 'label', 'group_id']
label
0    408800
1     40880
Name: count, dtype: int64
Unique groups: 40880


In [27]:
OUTPUT_RANKER_FILE = DATA_DIR / "ranker_training_dataset.csv"
ranker_df.to_csv(OUTPUT_RANKER_FILE, index=False)

print("Saved:", OUTPUT_RANKER_FILE)

Saved: D:\recommendation_item_API\data\ranker_training_dataset.csv


In [28]:
feature_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "isHoliday",
    "isFestival",
    "weekOfMonth"
]

target_col = "label"
group_col = "group_id"

print("Feature columns:", feature_cols)
print("Target column:", target_col)
print("Group column:", group_col)

Feature columns: ['basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'isHoliday', 'isFestival', 'weekOfMonth']
Target column: label
Group column: group_id
